# 03 · Líneas base y modelos candidatos
Experimento de MLflow: `resenas-bancos-ec`. Misma preparación (`src/pipeline.py`), misma validación (`TimeSeriesSplit(5)` sobre el train) y misma métrica (PR-AUC de la clase negativa) para todos.
Ejecutar con *Restart* y luego *Run All*. El conjunto de prueba **no se usa** aquí.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)          # test no se toca en este notebook
X, y = train[FEATURES], train[TARGET]
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()
print(X.shape, "| % negativa:", round(y.mean()*100, 1), "| corte del snapshot:", FECHA_CORTE)

## 1. Líneas base (2 corridas)
- **Ingenua:** `DummyClassifier` (prior de la clase).
- **De negocio:** regla de palabras de queja (lo que haría un analista a mano).

In [ ]:
limpiar_fase("baseline")
res = []
res.append(correr("baseline-dummy", Pipeline([("modelo", DummyClassifier(strategy="prior"))]),
                  "baseline", "dummy", X, y, {"strategy": "prior"}, FECHA_CORTE))
res.append(correr("baseline-regla-lexico", Pipeline([("modelo", ReglaNegocio())]),
                  "baseline", "regla_negocio", X, y, {"n_palabras_lexico": len(LEXICO_NEG)}, FECHA_CORTE))
pd.DataFrame(res)[["nombre","cv_pr_auc_mean","cv_pr_auc_std","cv_f1_mean","costo_cv_umbral"]].round(4)

## 2. Candidatos con hiperparámetros por defecto (4 corridas aquí; la 5.ª familia se agrega en `03b`)

In [ ]:
candidatos = {
    "lineal_regularizado": ("LogisticRegression", LogisticRegression(max_iter=1000), False),
    "bagging":             ("RandomForest",       RandomForestClassifier(random_state=SEED, n_jobs=-1), False),
    "boosting":            ("LightGBM",           LGBMClassifier(random_state=SEED, verbose=-1), False),
    "red_neuronal":        ("MLPClassifier",      MLPClassifier(random_state=SEED), False),
}
limpiar_fase("candidato", familias=list(candidatos))
for familia, (nombre, modelo, denso) in candidatos.items():
    print("→", nombre)
    res.append(correr(f"candidato-{nombre}", construir_pipeline(modelo, denso=denso), "candidato", familia,
                      X, y, {"modelo": nombre, **{k: str(v) for k, v in modelo.get_params().items() if k in ("C","n_estimators","learning_rate","hidden_layer_sizes","max_iter")}},
                      FECHA_CORTE))

## 3. Resultados

In [ ]:
t = pd.DataFrame(res).set_index("nombre")
cols = ["train_pr_auc","cv_pr_auc_mean","cv_pr_auc_std","brecha_pr_auc","cv_f1_mean","costo_cv_umbral","umbral_costo","segundos_entrenamiento"]
display(t[cols].round(4).sort_values("cv_pr_auc_mean", ascending=False))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
t["cv_pr_auc_mean"].sort_values().plot.barh(ax=ax[0], xerr=t["cv_pr_auc_std"].reindex(t["cv_pr_auc_mean"].sort_values().index), color="#6b7d3a")
ax[0].set_title("PR-AUC en CV (media ± desv.)")
t[["train_pr_auc","cv_pr_auc_mean"]].plot.bar(ax=ax[1], color=["#c9a227","#6b7d3a"]); ax[1].set_title("Entrenamiento vs CV (sobreajuste)")
plt.tight_layout(); plt.savefig("/work/docs/figs/candidatos.png", dpi=150); plt.show()

## 4. Diagnóstico (completar con las cifras de la tabla)
- ¿Algún modelo no supera la regla de negocio? → documentarlo como hallazgo.
- Brecha train − CV alta ⇒ sobreajuste (típico de RandomForest con TF-IDF).
- Los 3 mejores por PR-AUC en CV pasan a la fase de *tuning*.